# 🚦 Smart City Traffic Demand Prediction

---

## Objective

Develop a machine learning system capable of **accurately predicting traffic demand** (vehicles/hour) using historical traffic, road infrastructure, weather, and geolocation information.

## Models Used

| Model | Type | Key Strength |
|---|---|---|
| **Random Forest** | Bagging ensemble | Robust, low variance |
| **XGBoost** | Gradient boosting | High accuracy, regularisation |
| **LightGBM** | Leaf-wise gradient boosting | Speed + accuracy |
| **Weighted Ensemble** | Model averaging | Best overall generalisation |

## Evaluation Metrics

| Metric | Description |
|---|---|
| **R²** | Proportion of variance explained (higher = better) |
| **RMSE** | Root Mean Squared Error — penalises large errors |
| **MAE** | Mean Absolute Error — average magnitude of errors |
| **MAPE** | Mean Absolute Percentage Error — relative error (%) |

## Notebook Structure

1. Project Introduction  
2. Business Problem  
3. Dataset Description  
4. Import Libraries  
5. Load Dataset  
6. Exploratory Data Analysis  
7. Data Cleaning  
8. Feature Engineering  
9. Data Splitting  
10. Model Training  
11. Hyperparameter Tuning  
12. Ensemble Learning  
13. Model Evaluation  
14. Residual Analysis  
15. Feature Importance & SHAP  
16. Conclusion  
17. Future Improvements  

---
## 3. Import Libraries

In [ ]:
import os
import sys
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.gridspec import GridSpec
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Add project root to path
sys.path.append('..')
from src.preprocessing import preprocess_pipeline
from src.features import FeatureEngineer, get_model_features_list
from src.ensemble import WeightedEnsemble

warnings.filterwarnings('ignore')

# ── Global Plot Aesthetics ──────────────────────────────────────────────────
PALETTE   = "viridis"
PRIMARY   = "#4C72B0"
SECONDARY = "#DD8452"
ACCENT    = "#55A868"
RED       = "#C44E52"

sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams.update({
    'font.family'      : 'DejaVu Sans',
    'axes.labelsize'   : 13,
    'axes.titlesize'   : 15,
    'axes.titleweight' : 'bold',
    'figure.titlesize' : 18,
    'figure.titleweight': 'bold',
    'xtick.labelsize'  : 11,
    'ytick.labelsize'  : 11,
})

# ── Paths ───────────────────────────────────────────────────────────────────
DATA_PATH    = "../data/raw/smart_city_traffic_data.csv"
MODELS_DIR   = "../models"
REPORTS_DIR  = "../reports"
os.makedirs(os.path.join(REPORTS_DIR, 'figures'), exist_ok=True)

print("✅ Libraries imported successfully.")
print(f"   NumPy  : {np.__version__}")
print(f"   Pandas : {pd.__version__}")

In [ ]:
import os
import pandas as pd
os.chdir('..') # move to project root
from src.preprocessing import preprocess_pipeline
# Run preprocessing to get clean data
df_clean = preprocess_pipeline(
    file_path='data/raw/smart_city_traffic_data.csv',
    config_path='config/feature_config.yaml'
)
print("Clean data shape:", df_clean.shape)

---
## 7. Feature Engineering

Feature engineering is the **most impactful** step in this pipeline. Raw features alone cannot capture the rich temporal, spatial, and weather-interaction patterns that drive traffic demand.

All features are computed by [`src/features.py`](../src/features.py) in the `FeatureEngineer` class.

### 7.1 Rush Hour Indicator

**Formula:** `rush_hour_indicator = 1` if hour ∈ {7,8,9,16,17,18,19}, else `0`

**Why?** Traffic demand changes dramatically during commuting hours. A binary flag provides a crisp signal to the model that complements the continuous `hour` feature.

### 7.2 Weather Impact Score

**Formula:**
$$\text{weather\_impact\_score} = \underbrace{0.4 \times w_{\text{condition}}}_{\text{categorical severity}} + \underbrace{0.4 \times \frac{\text{rainfall}}{\text{max\_rainfall}}}_{\text{precipitation}} + \underbrace{0.2 \times \frac{\text{humidity}}{100}}_{\text{moisture}}$$

- $w_{\text{condition}}$: severity weight (Clear=0, Cloudy=1, Foggy=2, Rainy=3, Snowy=4, Stormy=5)

**Why?** A compound index captures the combined effect of weather severity better than any single feature alone.

### 7.3 Traffic Density Score

**Formula:**
$$\text{traffic\_density\_score} = \frac{\text{large\_vehicles\_count}}{\text{number\_of\_lanes} + 1} \times (1 + \text{traffic\_signals})$$

**Why?** Large vehicles disproportionately consume road space. Normalising by lanes and multiplying by signal count gives a **congestion proxy** that reflects how constrained the road infrastructure is.

### 7.4 Cyclical Time Encoding

**Why can't we encode `hour` directly?** Hours 23 and 0 are adjacent in time but *linearly far apart* (23 vs 0). A model using raw hour values would treat midnight as far from 11 PM.

**Solution:** Project onto the unit circle:
$$\text{hour\_sin} = \sin\left(\frac{2\pi \cdot h}{24}\right), \quad \text{hour\_cos} = \cos\left(\frac{2\pi \cdot h}{24}\right)$$

Applied to: **hour** (period=24), **day_of_week** (period=7), **month** (period=12).

### 7.5 Interaction Features

| Feature | Formula | Intuition |
|---|---|---|
| `rainfall_humidity_interaction` | `rainfall/max × humidity/100` | Combined precipitation intensity |
| `temp_wind_interaction` | `temp/max × wind/max` | Wind chill / heat compound |
| `vehicle_lane_ratio` | `large_veh / (lanes + 1)` | Congestion proxy |
| `rush_weekend_interaction` | `rush_hour × (1 - is_weekend)` | Weekday rush is distinct from weekend |
| `lane_signal_interaction` | `lanes × signals` | Infrastructure capacity |

### 7.6 Target Encodings (Spatio-temporal)

| Encoding | Key | Why |
|---|---|---|
| `location_encoded` | location | Location's historical mean demand |
| `road_type_encoded` | road_type | Road type's historical mean demand |
| `geohash_location_encoded` | geohash | Fine-grained spatial signal |
| `hour_location_encoded` | (hour, location) | Rush patterns vary by location |
| `dow_road_encoded` | (day_of_week, road_type) | Commuter patterns by road class |

> ⚠️ **Data Leakage Prevention:** Target encodings are fitted *only on the training set* and applied to validation/test sets using training statistics.

In [ ]:
# Demonstrate cyclical encoding
hours = np.arange(0, 24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Circle plot
axes[0].plot(hour_cos, hour_sin, 'o-', color=PRIMARY, lw=1.5, ms=4)
for i in [0, 6, 12, 18, 23]:
    axes[0].annotate(f'{i}h', (hour_cos[i]*1.08, hour_sin[i]*1.08),
                     ha='center', va='center', fontsize=10, color='black')
axes[0].set_aspect('equal')
axes[0].set_title('Cyclical Hour Encoding — Unit Circle')
axes[0].set_xlabel('cos(2π × hour / 24)')
axes[0].set_ylabel('sin(2π × hour / 24)')
axes[0].axhline(0, color='grey', lw=0.5)
axes[0].axvline(0, color='grey', lw=0.5)

# Wave plot
axes[1].plot(hours, hour_sin, label='sin (hour)', color=PRIMARY, lw=2)
axes[1].plot(hours, hour_cos, label='cos (hour)', color=SECONDARY, lw=2)
axes[1].axvspan(7, 9, alpha=0.15, color='gold', label='Morning Rush')
axes[1].axvspan(16, 19, alpha=0.15, color='tomato', label='Evening Rush')
axes[1].set_title('Sine & Cosine of Hour — Continuity at Midnight')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Encoded Value')
axes[1].set_xticks(range(0, 24))
axes[1].legend(fontsize=9)

plt.suptitle('Cyclical Time Encoding — Why hour 23 ≈ hour 0', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Apply feature engineering and show correlation of engineered features with target
fe_demo = FeatureEngineer()
fe_demo.fit(df_clean)
df_trans = fe_demo.transform(df_clean)

engineered_features = [
    'traffic_demand',
    'traffic_density_score', 'weather_impact_score',
    'rush_hour_indicator', 'hour_sin', 'hour_cos',
    'rainfall_humidity_interaction', 'temp_wind_interaction',
    'vehicle_lane_ratio', 'rush_weekend_interaction',
    'hour_location_encoded', 'dow_road_encoded'
]

corr_eng = df_trans[engineered_features].corr()
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    corr_eng, annot=True, fmt='.2f', cmap='coolwarm',
    linewidths=0.5, square=True, ax=ax, vmin=-1, vmax=1
)
ax.set_title('Engineered Feature Correlation Heatmap', pad=20)
plt.tight_layout()
plt.show()

# Correlation of each engineered feature with target
eng_target_corr = corr_eng['traffic_demand'].drop('traffic_demand').sort_values(key=abs, ascending=False)
print("\nEngineered feature correlations with traffic_demand:")
print(eng_target_corr.to_string())

---
## 8. Data Splitting

### Why chronological split?

This is a **time-series regression** problem. Using a random split would cause **temporal leakage** — future observations would appear in the training set, giving the model information it cannot have at inference time.

**Chronological split ensures:**
- The model trains only on past data
- Validation and test sets simulate real future deployments
- Reported metrics are honest estimates of production performance

### Split Proportions

```
───────────────────────────────────────────────────────
  2024                             →                2026
  ◄──────── TRAIN (70%) ────────►◄── VAL (15%) ──►◄── TEST (15%) ──►
  Fit models & target encodings    Tune weights    Final evaluation
───────────────────────────────────────────────────────
```

After tuning on the validation set, models are **retrained on Train + Val** combined before final test evaluation.

In [ ]:
# Reproduce the split
df_sorted = df_clean.sort_values('timestamp').reset_index(drop=True)
n = len(df_sorted)
train_end = int(n * 0.70)
val_end   = int(n * 0.85)

train_df = df_sorted.iloc[:train_end]
val_df   = df_sorted.iloc[train_end:val_end]
test_df  = df_sorted.iloc[val_end:]

print(f"📊 Chronological Split:")
print(f"   TRAIN : {len(train_df):,} rows  ({len(train_df)/n*100:.1f}%)  "
      f"[{train_df['timestamp'].min().date()} → {train_df['timestamp'].max().date()}]")
print(f"   VAL   : {len(val_df):,} rows  ({len(val_df)/n*100:.1f}%)  "
      f"[{val_df['timestamp'].min().date()} → {val_df['timestamp'].max().date()}]")
print(f"   TEST  : {len(test_df):,} rows  ({len(test_df)/n*100:.1f}%)  "
      f"[{test_df['timestamp'].min().date()} → {test_df['timestamp'].max().date()}]")

# Visualise split in time
fig, ax = plt.subplots(figsize=(14, 3))
for split, color, label in [
    (train_df, PRIMARY, 'Train (70%)'),
    (val_df,   ACCENT,  'Validation (15%)'),
    (test_df,  RED,     'Test (15%)')
]:
    monthly = split.set_index('timestamp').resample('ME')['traffic_demand'].mean()
    ax.plot(monthly.index, monthly.values, color=color, lw=2, label=label)

ax.axvline(train_df['timestamp'].max(), color='black', linestyle='--', lw=1.5, alpha=0.7)
ax.axvline(val_df['timestamp'].max(),   color='black', linestyle=':',  lw=1.5, alpha=0.7)
ax.set_title('Data Split — Monthly Mean Traffic Demand Over Time')
ax.set_xlabel('Date')
ax.set_ylabel('Mean Demand (veh/hr)')
ax.legend()
plt.tight_layout()
plt.show()